In [1]:
import pandas as pd
import numpy as np


from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

import altair as alt
import pickle

import os

In [2]:
import xgboost as xgb
import dill #aneto
import SuperLore
import category_encoders
import re
import json
import os

## Controregole-controfattuali

#### Carico il Modello

In [3]:
bb = xgb.XGBClassifier()
bb.load_model("../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_train.model")

In [4]:
bb

XGBClassifier(base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, early_stopping_rounds=50,
              enable_categorical=False, eval_metric='aucpr', feature_types=None,
              gamma=0.65, gpu_id=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.025, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=13, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              n_estimators=110, n_jobs=10, num_parallel_tree=None,
              objective='binary:hinge', predictor=None, ...)

#### Caricol x train, y train, x test, y test

In [10]:
X_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtrain')
Y_train = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytrain')
X_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xtest')
Y_test = pd.read_pickle('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_ytest')
data_desc=pd.read_pickle('../datasets/Dati-Banca-Lore/intesa_incassi_data_description.p')
path =('../datasets/Dati-Banca-Lore/INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_explanations_full.p')
explanations_shap = dill.load(open(path, 'rb'))

### DATA DESCRIPTION

In [6]:
type(data_desc)
data_desc

{'numerical': {'PCRIV_FT_20_DLT_PERC_ANNUO_UTILZZ_MEDIO': {'mean': 0.4963,
   'std': 0.1677,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.4274,
   '2nd-quantile': 0.5059,
   '3rd-quantile': 0.5592},
  'PCRIV_NAT_FT_02_LAST_YYYYMM_UTILIZZATO': {'mean': 0.4103,
   'std': 0.2236,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.246,
   '2nd-quantile': 0.3661,
   '3rd-quantile': 0.5307},
  'PRODV_LETTERE_DI_CREDITO_PON': {'mean': 0.5002,
   'std': 0.1472,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.453,
   '2nd-quantile': 0.453,
   '3rd-quantile': 0.453},
  'PN_OPS_WITH_SECTOR_FINANCIAL_TY_NUM': {'mean': 0.4061,
   'std': 0.2854,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.1745,
   '2nd-quantile': 0.3338,
   '3rd-quantile': 0.6341},
  'PN_SEPA_ENTRATA_TY_VAL': {'mean': 0.4587,
   'std': 0.2845,
   'min': 0.0,
   'max': 1.0,
   '1st-quantile': 0.2157,
   '2nd-quantile': 0.4202,
   '3rd-quantile': 0.7006},
  'SCADV_IMP_ESPOSIZIONE_ATTUALE_TOTALE': {'mean': 0.3707,
   's

#### Carico le spiegazioni di Lore

In [7]:
lore_exp_path=open('../datasets/Dati-Banca-Lore/lore_exp_INCASSI_E_PAGAMENTI_INTERNAZIONALI_27_xgb_cfs_binary_from_dts.p','rb')
objs = []
while 1:
    try:
        objs.append(pickle.load(lore_exp_path))
    except EOFError:
        break

In [8]:
for lore_p in objs[0].rule.premises:
    print(lore_p)

PN_SEPA_USCITA_TY_NUM = 1.00
SCADV_FLG_RATA_DIVISA_SEK = 1.00


### Carico SHAP

In [9]:
feature_names = X_test.columns
fi_shap={}
for j,f in enumerate(feature_names):
        fi_shap[f]= dict()
        fi_shap[f]['feature_importance'] = explanations_shap[0][j]
print(fi_shap)

NameError: name 'explanations_shap' is not defined

### CREO il dizionario

In [ ]:
rule_list=[]
def parse_rule(lore_obj): #parso le rule
    for lore_r in lore_obj.rule.premises:
        rule_list.append(vars(lore_r))    
    return rule_list

counterrule_list=[]
def parse_counterrules(lore_obj): #parso le counterrule
    for cr in lore_obj.crules:
        for nested in cr.premises:
            counterrule_list.append(vars(nested))
    return counterrule_list

exemplars_list = []
def parse_exemplars(lore_obj): #parso gli exemplars
    ex=lore_obj.exemplars
    new_ex = re.sub(r'(\d\.\d+)', r'\1 ', ex) #separo numeri da istanze
    pattern = r'{(.*?)}'
    matches = re.findall(pattern, new_ex, re.DOTALL)
    for match in matches:
        ex_parse = {}
        list_to_dict=[]
        properties = match.split() #ogni stringa separata da uno spazio diventa un nuovo item della lista, ogni lista è un exemplar
        for prop in properties: 
            if prop != "=": #elimino =
                list_to_dict.append(prop) # creo un dizionario con chiave e valore
        for i in range(0, len(list_to_dict), 2):
            ex_parse[list_to_dict[i]] = float(list_to_dict[i + 1])
        exemplars_list.append(ex_parse)
    return exemplars_list

dt_dict={}
def parse_dt(lore_obj): #parso dt, qui biasogna implementare il parsing dell'oggetto tree, che per ora sto sostiutendo con una stringa
    dt_dict=lore_obj.dt.__dict__
    dt_dict.update(
    {'tree_':'to add'
    })
    return dt_dict


def inst_value (n):
    inst= X_train.iloc[n]
    inst_dict = inst.to_dict()
    return inst_dict



In [ ]:
inst= X_train.iloc[0]
inst_dict= inst.to_dict()
print(inst_dict)

In [ ]:
def parse_dict_first(lore_obj): #creo un dizionario dell'istanza, alcuni oggetti però vanno parsati meglio
    istance = lore_obj.__dict__
    return istance

def update_object(istance):
    istance.update(
    {'rule': rule_list,
     'crules':counterrule_list,
     'exemplars':exemplars_list,
     'dt':dt_dict,
     'distribution':data_desc,
     'inst':inst_dict,
     'fi_shap':fi_shap
    })
    return istance   


In [ ]:
istance_0 = parse_dict_first(objs[0])
parse_rule(objs[0])
parse_counterrules(objs[0])
parse_exemplars(objs[0])
parse_dt(objs[0])
inst_value(0)

update_object(istance_0)
print(istance_0)

In [ ]:
istance_1 = parse_dict_first(objs[1])
parse_rule(objs[1])
parse_counterrules(objs[1])
parse_exemplars(objs[1])
parse_dt(objs[1])
inst_value(1)

update_object(istance_1)
print(istance_1)


In [ ]:
istance_2 = parse_dict_first(objs[2])
parse_rule(objs[2])
parse_counterrules(objs[2])
parse_exemplars(objs[2])
parse_dt(objs[2])
inst_value(2)

update_object(istance_2)
print(istance_2)

In [ ]:
istance_3 = parse_dict_first(objs[3])
parse_rule(objs[3])
parse_counterrules(objs[3])
parse_exemplars(objs[3])
parse_dt(objs[3])

update_object(istance_3)
print(istance_3)

In [ ]:
istance_4 = parse_dict_first(objs[4])
parse_rule(objs[4])
parse_counterrules(objs[4])
parse_exemplars(objs[4])
parse_dt(objs[4])

update_object(istance_4)
print(istance_4)

In [ ]:
istance_5 = parse_dict_first(objs[5])
parse_rule(objs[5])
parse_counterrules(objs[5])
parse_exemplars(objs[5])
parse_dt(objs[5])

update_object(istance_5)
print(istance_5)

#### ESPORTO IL JSON

In [ ]:
class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return json.JSONEncoder.default(self, obj)

In [ ]:
filename = "../JSON_istance/istance_0.json"
with open(filename, "w") as file:
    json.dump(istance_0, file, cls=NpEncoder)